# 01 · PreprocessingQC → normalize → HVG → PCA → Harmony → UMAP → cell-type harmonizationEach block is **Why → Test → Display**. Code is lifted from `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` with the source cell number recorded; anything not traceable to source is marked `⟨NEW⟩`.

## Configuration**Why.** One config module resolves every path and parameter from a single dataset key, so the only thing that differs between cohorts is that key. Merged from the dataset dicts already present in the source notebooks.

In [ ]:
from pathlib import Pathimport warnings; warnings.filterwarnings('ignore')from config import CFG, assert_layersimport config as CDATASET = 'psychad_aging'      # <<< the only line you changecfg = CFG.for_dataset(DATASET)cfg.echo()def why(block, question, rationale=None):    """Print the Why so it lands in executed output, not only in markdown."""    print("\n" + "=" * 78)    print(f"  {block}")    print("=" * 78)    print(f"  Q: {question}")    if rationale:        for line in rationale.split(" | "):            print(f"     {line}")    print()def gate(label, ok, detail=""):    """Fail loudly. A failed gate stops the module rather than flowing downstream."""    mark = "OK  " if ok else "FAIL"    print(f"  [{mark}] {label}{'  — ' + detail if detail else ''}")    if not ok:        raise AssertionError(f"GATE FAILED: {label}. {detail}")    return ok

## Setup**Why.** Pinned figure settings so panels are consistent across modules.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cells 2, 3</sub>

In [ ]:
why("Setup", "Pinned figure settings so panels are consistent across modules")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 2 ──import scanpy as scimport scanpy.external as sceimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathimport warningswarnings.filterwarnings('ignore')print(f"scanpy: {sc.__version__}")print(f"numpy: {np.__version__}")print(f"pandas: {pd.__version__}")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 3 ──# Publication-ready figure settingsplt.rcParams.update({    'figure.dpi': 150,    'savefig.dpi': 300,    'font.size': 10,    'axes.labelsize': 10,    'axes.titlesize': 11,    'legend.fontsize': 9,    'font.family': 'sans-serif',    'axes.linewidth': 1.0,    'axes.grid': False,    'pdf.fonttype': 42,})sc.settings.verbosity = 1sc.settings.set_figure_params(dpi=150, dpi_save=300, facecolor='white', frameon=False)print("✓ Figure settings configured")

## Load Data**Why.** Verifies the batch key exists before anything depends on it.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 7</sub>

In [ ]:
why("Load Data", "Verifies the batch key exists before anything depends on it")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 7 ──print("\n" + "="*80)print("LOADING DATA")print("="*80)adata = sc.read_h5ad(INPUT_FILE)print(f"\nLoaded: {INPUT_FILE.name}")print(f"  Cells: {adata.n_obs:,}")print(f"  Genes: {adata.n_vars:,}")print(f"  Sample genes: {list(adata.var_names[:5])}")# Verify batch key(s)batch_keys = BATCH_KEY if isinstance(BATCH_KEY, list) else [BATCH_KEY]for key in batch_keys:    if key in adata.obs.columns:        n_batches = adata.obs[key].nunique()        print(f"  Batch key '{key}': {n_batches} batches")    else:        raise ValueError(f"Batch key '{key}' not found in adata.obs")# Verify cell type columnif CELL_TYPE_COLUMN in adata.obs.columns:    n_types = adata.obs[CELL_TYPE_COLUMN].nunique()    print(f"  Cell type column '{CELL_TYPE_COLUMN}': {n_types} types")else:    print(f"  ⚠ Cell type column '{CELL_TYPE_COLUMN}' not found (will skip validation)")print("\n✓ Data loaded")

## Cell Type Harmonization**Why.** Cohorts label cell types differently (`subclass` vs `major_celltype`, shortened names). Mapping to one vocabulary is what makes cross-cohort comparison meaningful.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 9</sub>

In [ ]:
why("Cell Type Harmonization", "Cohorts label cell types differently (`subclass` vs `major_celltype`, shortened names)")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 9 ──print("\n" + "="*80)print("CELL TYPE HARMONIZATION")print("="*80)# Harmonization mapping for shortened namesHARMONIZATION_MAP = {    "Exc": "Excitatory", "Ex": "Excitatory", "Excitatory neurons": "Excitatory",    "Inh": "Inhibitory", "In": "Inhibitory", "Inhibitory neurons": "Inhibitory",    "Astro": "Astrocyte", "Astrocytes": "Astrocyte", "AST": "Astrocyte",    "Oligo": "Oligodendrocyte", "Olig": "Oligodendrocyte", "ODC": "Oligodendrocyte", "Oligodendrocytes": "Oligodendrocyte",    "Oligodendrocyte precursor": "OPC", "opc": "OPC", "OPCs": "OPC",    "Micro": "Microglia", "MG": "Microglia", "Mg": "Microglia",    "Vascular": "Vascular",    "Immune": "Immune",    "Endo": "Endothelial", "EC": "Endothelial",    "PC": "Pericyte", "Peri": "Pericyte",    "SMC": "VSMC",}# Harmonize the cell type columnif CELL_TYPE_COLUMN in adata.obs.columns:    print(f"\nHarmonizing {CELL_TYPE_COLUMN}...")        # Save original    adata.obs[f"{CELL_TYPE_COLUMN}_original"] = adata.obs[CELL_TYPE_COLUMN].copy()        # Convert to string if categorical    adata.obs[CELL_TYPE_COLUMN] = adata.obs[CELL_TYPE_COLUMN].astype(str)        original_types = adata.obs[CELL_TYPE_COLUMN].unique()    changes_made = []        # First: Apply direct mappings    for orig, std in HARMONIZATION_MAP.items():        mask = adata.obs[CELL_TYPE_COLUMN] == orig        if mask.any():            adata.obs.loc[mask, CELL_TYPE_COLUMN] = std            n_changed = mask.sum()            changes_made.append(f"  {orig} → {std} ({n_changed:,} cells)")        # Second: Collapse all EN_ subtypes to "Excitatory"    en_mask = adata.obs[CELL_TYPE_COLUMN].str.startswith('EN_')    if en_mask.any():        n_en = en_mask.sum()        adata.obs.loc[en_mask, CELL_TYPE_COLUMN] = 'Excitatory'        changes_made.append(f"  EN_* subtypes → Excitatory ({n_en:,} cells)")        # Third: Collapse all IN_ subtypes to "Inhibitory"    in_mask = adata.obs[CELL_TYPE_COLUMN].str.startswith('IN_')    if in_mask.any():        n_in = in_mask.sum()        adata.obs.loc[in_mask, CELL_TYPE_COLUMN] = 'Inhibitory'        changes_made.append(f"  IN_* subtypes → Inhibitory ({n_in:,} cells)")        if changes_made:        print("\nChanges made:")        for change in changes_made:            print(change)    else:        print("  No changes needed")        final_types = sorted(adata.obs[CELL_TYPE_COLUMN].unique())    print(f"\n  Original: {len(original_types)} types → Harmonized: {len(final_types)} types")        # Show all final cell types    type_counts = adata.obs[CELL_TYPE_COLUMN].value_counts()    print(f"\nFinal cell types:")    for cell_type, count in type_counts.items():        pct = count / len(adata.obs) * 100        print(f"  {cell_type}: {count:,} cells ({pct:.1f}%)")else:    print(f"\n⚠ {CELL_TYPE_COLUMN} column not found - skipping harmonization")print("\n✓ Cell type harmonization complete")

## Gene Conversion**Why.** ENSG IDs to symbols where needed; gene panels downstream are keyed on symbols.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 11</sub>

In [ ]:
why("Gene Conversion", "ENSG IDs to symbols where needed; gene panels downstream are keyed on symbols")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 11 ──if CONVERT_GENE_IDS:    print("\n" + "="*80)    print("GENE CONVERSION")    print("="*80)        sample_genes = adata.var_names[:10].tolist()    is_ensembl = any(g.startswith('ENS') for g in sample_genes)        if is_ensembl:        print(f"\nDetected Ensembl IDs")        print(f"Total genes: {len(adata.var_names):,}")                try:            import mygene        except ImportError:            print("\n⚠ mygene not installed. Run: pip install mygene")            raise                mg = mygene.MyGeneInfo()                chunk_size = 1000        all_mappings = {}        unconverted = []                print(f"Converting in chunks of {chunk_size}...")        for i in range(0, len(adata.var_names), chunk_size):            chunk = adata.var_names[i:i+chunk_size].tolist()            chunk_num = i//chunk_size + 1            total_chunks = (len(adata.var_names) - 1)//chunk_size + 1                        if chunk_num % 5 == 1:                print(f"  Progress: {chunk_num}/{total_chunks} chunks")                        results = mg.querymany(chunk, scopes='ensembl.gene', fields='symbol', species='human')                        for result in results:                if 'symbol' in result:                    all_mappings[result['query']] = result['symbol']                else:                    all_mappings[result['query']] = result['query']                    unconverted.append(result['query'])                new_names = [all_mappings.get(g, g) for g in adata.var_names]        n_converted = sum(1 for old, new in zip(adata.var_names, new_names) if old != new)                adata.var_names = pd.Index(new_names)        adata.var_names.name = 'gene_symbol'                # Handle duplicates        if adata.var_names.duplicated().any():            names = pd.Series(adata.var_names)            counts = names.groupby(names).cumcount()            names.loc[counts > 0] += '-' + counts.loc[counts > 0].astype(str)            adata.var_names = pd.Index(names.values)                print(f"\nConversion results:")        print(f"  Converted: {n_converted:,} ({n_converted/len(adata.var_names)*100:.1f}%)")        print(f"  Kept as Ensembl: {len(unconverted):,} ({len(unconverted)/len(adata.var_names)*100:.1f}%)")        print(f"  Sample genes: {list(adata.var_names[:5])}")        print("\n✓ Gene conversion complete")    else:        print(f"\nGenes already appear to be symbols")        print(f"Sample genes: {list(adata.var_names[:5])}")        print("Skipping conversion")else:    print("\nGene conversion skipped (CONVERT_GENE_IDS = False)")

## QC Metrics**Why.** Computes mitochondrial, ribosomal and haemoglobin fractions used by the filters.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 13</sub>

In [ ]:
why("QC Metrics", "Computes mitochondrial, ribosomal and haemoglobin fractions used by the filters")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 13 ──print("\n" + "="*80)print("QC METRICS")print("="*80)# Identify gene categoriesprint("\nIdentifying gene categories...")adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')adata.var['ribo'] = adata.var_names.str.upper().str.match('^RP[SL]')adata.var['hgb'] = adata.var_names.str.upper().str.contains('^HB[AB]')n_mt = adata.var['mt'].sum()n_ribo = adata.var['ribo'].sum()n_hgb = adata.var['hgb'].sum()print(f"  MT genes: {n_mt}")print(f"  Ribosomal genes: {n_ribo}")print(f"  Hemoglobin genes: {n_hgb}")# Calculate metricsprint("\nCalculating QC metrics...")sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo', 'hgb'], inplace=True)print(f"\nMedian QC metrics:")print(f"  Genes/cell: {adata.obs['n_genes_by_counts'].median():.0f}")print(f"  Counts/cell: {adata.obs['total_counts'].median():.0f}")print(f"  MT%: {adata.obs['pct_counts_mt'].median():.2f}")print(f"  Ribo%: {adata.obs['pct_counts_ribo'].median():.2f}")print(f"  HGB%: {adata.obs['pct_counts_hgb'].median():.2f}")print("\n✓ QC metrics calculated")

## QC Visualization**Why.** Inspect distributions before choosing thresholds, not after.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 15</sub>

In [ ]:
why("QC Visualization", "Inspect distributions before choosing thresholds, not after")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 15 ──# 6-panel QC plotfig, axes = plt.subplots(2, 3, figsize=(12, 8))for ax in axes.flat:    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.grid(alpha=0.3, linewidth=0.5)# Genes per cellaxes[0,0].hist(adata.obs['n_genes_by_counts'], bins=50, color='#2E86AB', alpha=0.7, edgecolor='white', linewidth=0.5)axes[0,0].axvline(MIN_GENES, color='#E63946', linestyle='--', linewidth=1.5, alpha=0.8)axes[0,0].axvline(MAX_GENES, color='#E63946', linestyle='--', linewidth=1.5, alpha=0.8)axes[0,0].set_xlabel('Genes per cell')axes[0,0].set_ylabel('Number of cells')axes[0,0].set_title('Genes Detected', fontsize=10, pad=10)# Total countsaxes[0,1].hist(np.log10(adata.obs['total_counts']), bins=50, color='#06A77D', alpha=0.7, edgecolor='white', linewidth=0.5)axes[0,1].set_xlabel('log₁₀(Total counts)')axes[0,1].set_ylabel('Number of cells')axes[0,1].set_title('Total Counts', fontsize=10, pad=10)# MT%axes[0,2].hist(adata.obs['pct_counts_mt'], bins=50, color='#F77F00', alpha=0.7, edgecolor='white', linewidth=0.5)axes[0,2].axvline(MAX_MT_PCT, color='#E63946', linestyle='--', linewidth=1.5, alpha=0.8)axes[0,2].set_xlabel('% Mitochondrial')axes[0,2].set_ylabel('Number of cells')axes[0,2].set_title('MT %', fontsize=10, pad=10)# Ribo%axes[1,0].hist(adata.obs['pct_counts_ribo'], bins=50, color='#9B59B6', alpha=0.7, edgecolor='white', linewidth=0.5)axes[1,0].set_xlabel('% Ribosomal')axes[1,0].set_ylabel('Number of cells')axes[1,0].set_title('Ribosomal %', fontsize=10, pad=10)# HGB%axes[1,1].hist(adata.obs['pct_counts_hgb'], bins=50, color='#E76F51', alpha=0.7, edgecolor='white', linewidth=0.5)axes[1,1].set_xlabel('% Hemoglobin')axes[1,1].set_ylabel('Number of cells')axes[1,1].set_title('Hemoglobin %', fontsize=10, pad=10)# Genes vs Counts scatterscatter = axes[1,2].scatter(adata.obs['total_counts'], adata.obs['n_genes_by_counts'],                            c=adata.obs['pct_counts_mt'], s=0.5, alpha=0.3, cmap='viridis', rasterized=True)axes[1,2].set_xlabel('Total counts')axes[1,2].set_ylabel('Genes detected')axes[1,2].set_title('Genes vs Counts', fontsize=10, pad=10)axes[1,2].set_xscale('log')axes[1,2].set_yscale('log')cbar = plt.colorbar(scatter, ax=axes[1,2], fraction=0.046, pad=0.04)cbar.set_label('MT %', rotation=270, labelpad=15)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_qc_metrics.svg', dpi=300, bbox_inches='tight')plt.show()print("✓ QC plots saved")

## QC Filtering**Why.** Applied after inspection. Counting cells failing each threshold separately shows which filter is doing the work — a single combined count hides a filter that removes almost everything.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 17</sub>

In [ ]:
why("QC Filtering", "Applied after inspection")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 17 ──print("\n" + "="*80)print("QC FILTERING")print("="*80)n_cells_before = adata.n_obsn_genes_before = adata.n_varsprint(f"\nBefore filtering:")print(f"  Cells: {n_cells_before:,}")print(f"  Genes: {n_genes_before:,}")# Count cells failing each thresholdfail_min_genes = (adata.obs['n_genes_by_counts'] < MIN_GENES).sum()fail_max_genes = (adata.obs['n_genes_by_counts'] > MAX_GENES).sum()fail_min_counts = (adata.obs['total_counts'] < MIN_COUNTS).sum()fail_mt = (adata.obs['pct_counts_mt'] > MAX_MT_PCT).sum()fail_ribo = (adata.obs['pct_counts_ribo'] > MAX_RIBO_PCT).sum()fail_hgb = (adata.obs['pct_counts_hgb'] > MAX_HGB_PCT).sum()print(f"\nCells failing thresholds:")print(f"  Too few genes (<{MIN_GENES}): {fail_min_genes:,}")print(f"  Too many genes (>{MAX_GENES}): {fail_max_genes:,}")print(f"  Too few counts (<{MIN_COUNTS}): {fail_min_counts:,}")print(f"  High MT% (>{MAX_MT_PCT}): {fail_mt:,}")print(f"  High Ribo% (>{MAX_RIBO_PCT}): {fail_ribo:,}")print(f"  High HGB% (>{MAX_HGB_PCT}): {fail_hgb:,}")# Apply filtersprint(f"\nApplying filters...")adata = adata[    (adata.obs['n_genes_by_counts'] >= MIN_GENES) &    (adata.obs['n_genes_by_counts'] <= MAX_GENES) &    (adata.obs['total_counts'] >= MIN_COUNTS) &    (adata.obs['pct_counts_mt'] <= MAX_MT_PCT) &    (adata.obs['pct_counts_ribo'] <= MAX_RIBO_PCT) &    (adata.obs['pct_counts_hgb'] <= MAX_HGB_PCT), :].copy()# Filter genessc.pp.filter_genes(adata, min_cells=MIN_CELLS)n_cells_after = adata.n_obsn_genes_after = adata.n_varsprint(f"\nAfter filtering:")print(f"  Cells: {n_cells_after:,} (removed {n_cells_before-n_cells_after:,}, {(n_cells_before-n_cells_after)/n_cells_before*100:.1f}%)")print(f"  Genes: {n_genes_after:,} (removed {n_genes_before-n_genes_after:,}, {(n_genes_before-n_genes_after)/n_genes_before*100:.1f}%)")print("\n✓ Filtering complete")

## Normalization**Why.** Log-normalization for visualization and module scoring. This is NOT the matrix SenePy scores on — module 02 produces that separately.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 19</sub>

In [ ]:
why("Normalization", "Log-normalization for visualization and module scoring")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 19 ──print("\n" + "="*80)print("NORMALIZATION")print("="*80)print("\n1. Saving raw counts...")adata.layers['counts'] = adata.X.copy()print("   ✓ layers['counts']")print(f"\n2. Normalizing to {TARGET_SUM:,} counts/cell...")sc.pp.normalize_total(adata, target_sum=TARGET_SUM)print("   ✓ Normalized")print("\n3. Log transforming...")sc.pp.log1p(adata)print("   ✓ Log-transformed")print("\n✓ Normalization complete")print("  adata.X: log-normalized expression")print("  layers['counts']: raw counts (for DESeq2)")

## UMI Distribution QC**Why.** Baseline depth distribution, recorded before any correction so module 02's effect can be measured against it.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 21</sub>

In [ ]:
why("UMI Distribution QC", "Baseline depth distribution, recorded before any correction so module 02's effect can be measured against it")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 21 ──# ═══════════════════════════════════════════════════════════════════════════════# UMI DISTRIBUTION QC (Baseline - Pre-Analysis)# ═══════════════════════════════════════════════════════════════════════════════print("\n" + "="*80)print("UMI DISTRIBUTION QC")print("="*80)from scipy.stats import spearmanr, pearsonr# StyleCELL_TYPE_COLORS = {    'Excitatory': '#4E79A7', 'Inhibitory': '#F28E2B', 'Astrocyte': '#E15759',    'Oligodendrocyte': '#76B7B2', 'OPC': '#59A14F', 'Microglia': '#EDC948',    'Endothelial': '#B07AA1', 'Pericyte': '#FF9DA7', 'VSMC': '#9C755F',    'VLMC': '#BAB0AC', 'PVM': '#D37295', 'Adaptive': '#FABFD2',    'Vascular': '#D37295', 'Immune': '#FABFD2',}umi = adata.obs['total_counts']genes = adata.obs['n_genes_by_counts']# ─────────────────────────────────────────────────────────────────────────────# A. Overall Statistics# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("A. OVERALL STATISTICS")print(f"{'─'*60}")print(f"\n  {'Metric':<20} {'UMI':>12} {'Genes':>12}")print(f"  {'-'*46}")print(f"  {'N Cells':<20} {len(umi):>12,} {len(genes):>12,}")print(f"  {'Min':<20} {umi.min():>12,.0f} {genes.min():>12,.0f}")print(f"  {'Q25':<20} {umi.quantile(0.25):>12,.0f} {genes.quantile(0.25):>12,.0f}")print(f"  {'Median':<20} {umi.median():>12,.0f} {genes.median():>12,.0f}")print(f"  {'Q75':<20} {umi.quantile(0.75):>12,.0f} {genes.quantile(0.75):>12,.0f}")print(f"  {'Max':<20} {umi.max():>12,.0f} {genes.max():>12,.0f}")print(f"  {'IQR':<20} {umi.quantile(0.75) - umi.quantile(0.25):>12,.0f} {genes.quantile(0.75) - genes.quantile(0.25):>12,.0f}")cv_umi = umi.std() / umi.mean()cv_genes = genes.std() / genes.mean()print(f"  {'CV (std/mean)':<20} {cv_umi:>12.2f} {cv_genes:>12.2f}")# ─────────────────────────────────────────────────────────────────────────────# B. UMI × Genes Correlation# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("B. UMI × GENES CORRELATION")print(f"{'─'*60}")corr_spearman, p_spearman = spearmanr(umi, genes)corr_pearson, p_pearson = pearsonr(umi, genes)print(f"\n  Overall correlation:")print(f"    Spearman ρ = {corr_spearman:.3f} (p = {p_spearman:.2e})")print(f"    Pearson r  = {corr_pearson:.3f} (p = {p_pearson:.2e})")if corr_spearman > 0.8:    print(f"\n  ⚠ Very high correlation (ρ={corr_spearman:.2f}) — UMI strongly drives gene detection")    print(f"    → Include total_counts as covariate in downstream models")# ─────────────────────────────────────────────────────────────────────────────# C. Overall Plots# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("C. OVERALL PLOTS")print(f"{'─'*60}")fig, axes = plt.subplots(1, 3, figsize=(10, 3))for ax in axes:    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)# Panel 1: UMI histogramaxes[0].hist(np.log10(umi + 1), bins=50, color='#55A868', alpha=0.7, edgecolor='white', linewidth=0.3)axes[0].axvline(np.log10(umi.median() + 1), color='#C44E52', linestyle='--', linewidth=1.5)axes[0].text(0.97, 0.95, f'median: {umi.median():,.0f}', transform=axes[0].transAxes,             fontsize=8, ha='right', va='top', color='#C44E52')axes[0].set_xlabel('log₁₀(UMI + 1)')axes[0].set_ylabel('Cells')axes[0].set_title('UMI Distribution')# Panel 2: Genes histogramaxes[1].hist(genes, bins=50, color='#4E79A7', alpha=0.7, edgecolor='white', linewidth=0.3)axes[1].axvline(genes.median(), color='#C44E52', linestyle='--', linewidth=1.5)axes[1].text(0.97, 0.95, f'median: {genes.median():,.0f}', transform=axes[1].transAxes,             fontsize=8, ha='right', va='top', color='#C44E52')axes[1].set_xlabel('Genes detected')axes[1].set_ylabel('Cells')axes[1].set_title('Gene Detection Distribution')# Panel 3: UMI × Genes scatteraxes[2].scatter(umi, genes, s=1, alpha=0.1, c='#555555', rasterized=True)axes[2].set_xlabel('UMI counts')axes[2].set_ylabel('Genes detected')axes[2].set_title(f'UMI × Genes (ρ = {corr_spearman:.2f})')# Add trend linez = np.polyfit(umi, genes, 1)p = np.poly1d(z)x_line = np.linspace(umi.min(), umi.max(), 100)axes[2].plot(x_line, p(x_line), 'r--', linewidth=1.5, alpha=0.8)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_genes_overall.svg', dpi=300, bbox_inches='tight')plt.show()print(f"\n✓ Saved: {DATASET}_umi_genes_overall.svg")# ─────────────────────────────────────────────────────────────────────────────# D. Per Cell Type Statistics# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("D. PER CELL TYPE STATISTICS")print(f"{'─'*60}")ct_order = adata.obs[CELL_TYPE_COLUMN].value_counts().index.tolist()print(f"\n  {'Cell Type':<16} {'N Cells':>10} {'Med UMI':>10} {'Med Genes':>10} {'UMI×Gene ρ':>12}")print(f"  {'-'*62}")ct_stats = []for ct in ct_order:    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct    ct_umi = adata.obs.loc[ct_mask, 'total_counts']    ct_genes = adata.obs.loc[ct_mask, 'n_genes_by_counts']    ct_corr, _ = spearmanr(ct_umi, ct_genes)        ct_stats.append({        'Cell_Type': ct,        'N': len(ct_umi),        'Median_UMI': ct_umi.median(),        'Median_Genes': ct_genes.median(),        'Corr': ct_corr    })        print(f"  {ct:<16} {len(ct_umi):>10,} {ct_umi.median():>10,.0f} {ct_genes.median():>10,.0f} {ct_corr:>12.3f}")df_ct_stats = pd.DataFrame(ct_stats)# ─────────────────────────────────────────────────────────────────────────────# E. Per Cell Type Plots# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("E. PER CELL TYPE PLOTS")print(f"{'─'*60}")n_ct = len(ct_order)# E1: Violin plots (UMI and Genes side by side)fig, axes = plt.subplots(1, 2, figsize=(max(7, n_ct * 0.6), 3.5))for ax in axes:    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)# UMI violinumi_data = [np.log10(adata.obs[adata.obs[CELL_TYPE_COLUMN] == ct]['total_counts'].values + 1)             for ct in ct_order]vp1 = axes[0].violinplot(umi_data, positions=range(n_ct), showmedians=True, showextrema=False)for i, (body, ct) in enumerate(zip(vp1['bodies'], ct_order)):    color = CELL_TYPE_COLORS.get(ct, '#808080')    body.set_facecolor(color)    body.set_alpha(0.7)    body.set_edgecolor(color)vp1['cmedians'].set_color('#333333')vp1['cmedians'].set_linewidth(1.5)axes[0].set_xticks(range(n_ct))axes[0].set_xticklabels(ct_order, rotation=45, ha='right', fontsize=8)axes[0].set_ylabel('log₁₀(UMI + 1)')axes[0].set_title('UMI by Cell Type')# Genes violingenes_data = [adata.obs[adata.obs[CELL_TYPE_COLUMN] == ct]['n_genes_by_counts'].values               for ct in ct_order]vp2 = axes[1].violinplot(genes_data, positions=range(n_ct), showmedians=True, showextrema=False)for i, (body, ct) in enumerate(zip(vp2['bodies'], ct_order)):    color = CELL_TYPE_COLORS.get(ct, '#808080')    body.set_facecolor(color)    body.set_alpha(0.7)    body.set_edgecolor(color)vp2['cmedians'].set_color('#333333')vp2['cmedians'].set_linewidth(1.5)axes[1].set_xticks(range(n_ct))axes[1].set_xticklabels(ct_order, rotation=45, ha='right', fontsize=8)axes[1].set_ylabel('Genes detected')axes[1].set_title('Genes by Cell Type')plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_genes_violin_by_celltype.svg', dpi=300, bbox_inches='tight')plt.show()print(f"\n✓ Saved: {DATASET}_umi_genes_violin_by_celltype.svg")# E2: UMI × Genes scatter faceted by cell typen_cols = 4n_rows = int(np.ceil(n_ct / n_cols))fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.2))axes = axes.flatten()for idx, ct in enumerate(ct_order):    ax = axes[idx]        ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct    ct_umi = adata.obs.loc[ct_mask, 'total_counts']    ct_genes = adata.obs.loc[ct_mask, 'n_genes_by_counts']    ct_corr = df_ct_stats[df_ct_stats['Cell_Type'] == ct]['Corr'].values[0]        color = CELL_TYPE_COLORS.get(ct, '#808080')    ax.scatter(ct_umi, ct_genes, s=1, alpha=0.2, c=color, rasterized=True)        ax.set_title(f'{ct} (ρ={ct_corr:.2f})', fontsize=9, color=color, fontweight='medium')    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.tick_params(labelsize=7)        if idx >= (n_rows - 1) * n_cols:        ax.set_xlabel('UMI', fontsize=8)    if idx % n_cols == 0:        ax.set_ylabel('Genes', fontsize=8)# Hide empty axesfor idx in range(n_ct, len(axes)):    axes[idx].set_visible(False)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_genes_scatter_by_celltype.svg', dpi=300, bbox_inches='tight')plt.show()print(f"✓ Saved: {DATASET}_umi_genes_scatter_by_celltype.svg")print("\n" + "="*80)print("✓ UMI DISTRIBUTION QC COMPLETE")print("="*80)

## Highly Variable Genes (HVG)**Why.** HVGs are selected on raw counts, not on the normalized matrix.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 23</sub>

In [ ]:
why("Highly Variable Genes (HVG)", "HVGs are selected on raw counts, not on the normalized matrix")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 23 ──print("\n" + "="*80)print("HIGHLY VARIABLE GENES")print("="*80)print(f"\nSelecting top {N_TOP_GENES} HVGs (seurat_v3)...")sc.pp.highly_variable_genes(    adata,    n_top_genes=N_TOP_GENES,    flavor='seurat_v3',    layer='counts',    subset=False)n_hvg = adata.var['highly_variable'].sum()print(f"\n✓ Selected {n_hvg:,} HVGs")print("\nSaving full matrix to adata.raw...")# Create a temporary copy with raw counts as .Xadata_temp = adata.copy()adata_temp.X = adata.layers['counts'].copy()# Now save to adata.raw (this works!)adata.raw = adata_tempdel adata_tempprint(f"  adata.raw: {adata.raw.n_obs:,} cells × {adata.raw.n_vars:,} genes")print(f"  adata.raw.X: raw counts (all genes)")print("\n✓ HVG selection complete")

## HVG Visualization**Why.** Confirms the mean-variance relationship looks sane before PCA.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 25</sub>

In [ ]:
why("HVG Visualization", "Confirms the mean-variance relationship looks sane before PCA")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 25 ──# HVG plot with fixed axis limitsfig, ax = plt.subplots(figsize=(8, 5))hvg_mask = adata.var['highly_variable'].values# Get mean and dispersion valuesif 'means' in adata.var.columns:    means = adata.var['means'].valueselif 'mean_counts' in adata.var.columns:    means = adata.var['mean_counts'].valueselse:    means = np.asarray(adata.layers['counts'].mean(axis=0)).flatten()if 'dispersions' in adata.var.columns:    dispersions = adata.var['dispersions'].valueselif 'dispersions_norm' in adata.var.columns:    dispersions = adata.var['dispersions_norm'].valueselif 'variances' in adata.var.columns:    dispersions = adata.var['variances'].valueselif 'variances_norm' in adata.var.columns:    dispersions = adata.var['variances_norm'].valueselse:    dispersions = np.ones(len(means))# Plotax.scatter(means[~hvg_mask], dispersions[~hvg_mask], s=3, alpha=0.3, color='lightgray',            label='Not highly variable', rasterized=True)ax.scatter(means[hvg_mask], dispersions[hvg_mask], s=3, alpha=0.6, color='#E63946',            label='Highly variable', rasterized=True)ax.set_xlabel('Mean expression')ax.set_ylabel('Dispersion')ax.set_title(f'Highly Variable Genes (n={hvg_mask.sum():,})', fontsize=11, pad=10)ax.legend(markerscale=3, frameon=False)ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)ax.grid(alpha=0.3, linewidth=0.5)# Fixed axis limits for better visualizationax.set_xlim(0, 40)ax.set_ylim(0, 3500)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_hvg_selection.svg', dpi=300, bbox_inches='tight')plt.show()print("✓ HVG plot saved")

## Scaling & PCA**Why.** Linear dimensionality reduction ahead of batch correction.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cells 27, 28</sub>

In [ ]:
why("Scaling & PCA", "Linear dimensionality reduction ahead of batch correction")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 27 ──print("\n1. Regressing out confounders...")print("   ✓ Confounders regressing")sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])print("   ✓ Confounders regressed")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 28 ──print("\n" + "="*80)print("SCALING & PCA")print("="*80)print(f"\n2. Subsetting to {adata.var['highly_variable'].sum():,} HVGs...")adata = adata[:, adata.var['highly_variable']].copy()print(f"   ✓ adata.X: {adata.n_obs:,} × {adata.n_vars:,}")print("\n3. Scaling (max value=10)...")sc.pp.scale(adata, max_value=10)adata.layers['scaled'] = adata.X.copy()print("   ✓ layers['scaled']")print(f"\n4. Computing PCA ({N_PCS} components)...")sc.tl.pca(adata, n_comps=N_PCS, svd_solver='arpack')var_explained = adata.uns['pca']['variance_ratio'][:N_PCS].sum()print(f"   ✓ PCA complete: {var_explained:.1%} variance explained")print("\n✓ Scaling & PCA complete")

## PCA Variance**Why.** Component selection — how many PCs actually carry signal.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 30</sub>

In [ ]:
why("PCA Variance", "Component selection — how many PCs actually carry signal")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 30 ──# PCA variance plotsc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)print("✓ PCA variance plot displayed")

## Batch Correction (Harmony)**Why.** Corrects on the batch key from config. Cross-cohort integration happens here; downstream models still carry Cohort as a covariate, because embedding correction and model adjustment address different things.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 32</sub>

In [ ]:
why("Batch Correction (Harmony)", "Corrects on the batch key from config")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 32 ──print("\n" + "="*80)print("HARMONY BATCH CORRECTION")print("="*80)n_batches = adata.obs[BATCH_KEY].nunique()print(f"\nRunning Harmony on {n_batches} batches ({BATCH_KEY})...")sce.pp.harmony_integrate(    adata,    key=BATCH_KEY,    basis='X_pca',    max_iter_harmony=20,    adjusted_basis='X_pca_harmony')print("\n✓ Harmony complete")print("  adata.obsm['X_pca_harmony']: batch-corrected PCs")

## UMAP & Clustering**Why.** Embedding for visualization and the clusters used by later subsetting.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 34</sub>

In [ ]:
why("UMAP & Clustering", "Embedding for visualization and the clusters used by later subsetting")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 34 ──print("\n" + "="*80)print("UMAP & CLUSTERING")print("="*80)print(f"\n1. Computing neighbors (k={N_NEIGHBORS})...")sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, use_rep='X_pca_harmony')print("   ✓ Neighbor graph computed")print("\n2. Computing UMAP...")sc.tl.umap(adata)print("   ✓ UMAP complete")print(f"\n3. Leiden clustering (resolution={LEIDEN_RESOLUTION})...")sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION)n_clusters = adata.obs['leiden'].nunique()print(f"   ✓ {n_clusters} clusters identified")print("\n✓ UMAP & clustering complete")

## Visualizations**Why.** Batch, cluster, cell type and QC overlays — the check that correction worked.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cells 36, 37, 38</sub>

In [ ]:
why("Visualizations", "Batch, cluster, cell type and QC overlays — the check that correction worked")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 36 ──# UMAP overview: batches and clustersfig, axes = plt.subplots(1, 2, figsize=(14, 6))sc.pl.umap(adata, color=BATCH_KEY[0], ax=axes[0], show=False, title='Batches',            frameon=False, size=8, legend_loc=None)axes[0].set_xlabel('UMAP 1')axes[0].set_ylabel('UMAP 2')sc.pl.umap(adata, color='leiden', ax=axes[1], show=False, legend_loc='on data',            title='Leiden Clusters', frameon=False, size=8)axes[1].set_xlabel('UMAP 1')axes[1].set_ylabel('')plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umap_overview.svg', dpi=300, bbox_inches='tight')plt.show()print("✓ UMAP overview saved")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 37 ──# UMAP colored by harmonized cell typesfig, ax = plt.subplots(figsize=(10, 8))sc.pl.umap(adata, color=CELL_TYPE_COLUMN, ax=ax, show=False,            title='Cell Types (Harmonized)', frameon=False, size=8,            legend_loc='right margin')ax.set_xlabel('UMAP 1')ax.set_ylabel('UMAP 2')plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umap_celltypes.svg', dpi=300, bbox_inches='tight')plt.show()print("✓ UMAP cell types saved")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 38 ──# UMAP colored by QC metricsfig, axes = plt.subplots(1, 3, figsize=(14, 4))sc.pl.umap(adata, color='pct_counts_mt', ax=axes[0], show=False,            title='MT %', frameon=False, size=2)axes[0].set_xlabel('UMAP 1')axes[0].set_ylabel('UMAP 2')sc.pl.umap(adata, color='total_counts', ax=axes[1], show=False,            title='Total Counts', frameon=False, size=2)axes[1].set_xlabel('UMAP 1')axes[1].set_ylabel('')sc.pl.umap(adata, color='n_genes_by_counts', ax=axes[2], show=False,            title='Genes Detected', frameon=False, size=2)axes[2].set_xlabel('UMAP 1')axes[2].set_ylabel('')plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umap_qc.svg', dpi=300, bbox_inches='tight')plt.show()print("✓ UMAP QC metrics saved")

## Save Results**Why.** Writes `{dataset}_preprocessed.h5ad` with `.X` log-normalized and `layers['counts']` raw. Module 02 requires both.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cells 40, 41</sub>

In [ ]:
why("Save Results", "Writes `{dataset}_preprocessed")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 40 ──print("\n" + "="*80)print("SAVING")print("="*80)import scipy.sparse as spprint("\nPreparing data for saving...")print(f"  Current adata: {adata.n_obs:,} cells × {adata.n_vars:,} genes (HVGs only)")print(f"  adata.raw: {adata.raw.n_obs:,} cells × {adata.raw.n_vars:,} genes (all genes)")# ──────────────────────────────────────────────────────────────────────────────# 1. Restore full gene set from adata.raw# ──────────────────────────────────────────────────────────────────────────────print("\nRestoring all genes from adata.raw...")adata_full = adata.raw.to_adata()# Verify what .raw contains (sparse-safe)if sp.issparse(adata_full.X):    raw_min = adata_full.X.data.min() if len(adata_full.X.data) > 0 else 0    raw_max = adata_full.X.data.max() if len(adata_full.X.data) > 0 else 0    sample_data = np.array(adata_full.X.data[:1000])    is_integer = np.allclose(sample_data, np.round(sample_data))else:    raw_min = float(np.min(adata_full.X))    raw_max = float(np.max(adata_full.X))    sample_data = np.array(adata_full.X.flat[:1000])    is_integer = np.allclose(sample_data, np.round(sample_data))print(f"  Raw X range: [{raw_min:.3f}, {raw_max:.3f}]")print(f"  Contains integers (raw counts): {is_integer}")# ──────────────────────────────────────────────────────────────────────────────# 2. Store raw counts# ──────────────────────────────────────────────────────────────────────────────print("\n  → layers['counts'] = raw counts")adata_full.layers['counts'] = adata_full.X.copy()# ──────────────────────────────────────────────────────────────────────────────# 3. Log-normalize .X# ──────────────────────────────────────────────────────────────────────────────if 'log1p' in adata_full.uns:    del adata_full.uns['log1p']print("  → Log-normalizing .X...")sc.pp.normalize_total(adata_full, target_sum=TARGET_SUM)sc.pp.log1p(adata_full)print("  ✓ .X = log-normalized expression")# ──────────────────────────────────────────────────────────────────────────────# 4. Transfer embeddings, clustering, and metadata# ──────────────────────────────────────────────────────────────────────────────print("\nTransferring embeddings and clustering...")adata_full.obsm = adata.obsm.copy()adata_full.obsp = adata.obsp.copy()adata_full.uns = adata.uns.copy()adata_full.obs = adata.obs.copy()# Mark HVGsadata_full.var['highly_variable'] = Falseadata_full.var.loc[adata.var_names, 'highly_variable'] = True# ──────────────────────────────────────────────────────────────────────────────# 5. Summary and save# ──────────────────────────────────────────────────────────────────────────────print(f"\nFinal data structure:")print(f"  Cells: {adata_full.n_obs:,}")print(f"  Genes: {adata_full.n_vars:,}")print(f"  adata.X: log-normalized expression")print(f"  adata.layers['counts']: raw counts (untouched)")print(f"  adata.var['highly_variable']: {adata_full.var['highly_variable'].sum():,} HVGs")OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)print(f"\nSaving to: {OUTPUT_FILE}")adata_full.write_h5ad(OUTPUT_FILE)file_size = OUTPUT_FILE.stat().st_size / 1e9print(f"\n✓ Saved ({file_size:.2f} GB)")# Update referenceadata = adata_full

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 41 ──print(type(adata_full.X))print(adata_full.X.shape)print(adata_full.X.dtype)# Check a small sliceimport scipy.sparse as spif sp.issparse(adata_full.X):    sample = adata_full.X[:5, :5].toarray()else:    sample = adata_full.X[:5, :5]print(sample)print(f"Min: {sample.min()}, Max: {sample.max()}")

## Summary**Why.** Run record.<sub>source: `00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb` cell 43</sub>

In [ ]:
why("Summary", "Run record")

In [ ]:
# ── source: 00_preprocessing_v4_psychencode_aging_regressedCopy1.ipynb cell 43 ──print("\n" + "="*80)print("✓ PREPROCESSING COMPLETE")print("="*80)print(f"\nDataset: {DATASET}")print(f"\nFinal dimensions:")print(f"  Cells: {adata.n_obs:,}")print(f"  Genes: {adata.n_vars:,}")print(f"  Clusters: {adata.obs['leiden'].nunique()}")print(f"  Cell types: {adata.obs[CELL_TYPE_COLUMN].nunique()}")print(f"\nData structure:")print(f"  adata.X: log-normalized expression")print(f"  adata.layers['counts']: raw counts")print(f"  adata.var['highly_variable']: {adata.var['highly_variable'].sum():,} HVGs")print(f"\nEmbeddings:")print(f"  adata.obsm['X_pca']: PCA ({N_PCS} PCs)")print(f"  adata.obsm['X_pca_harmony']: Harmony-corrected PCA")print(f"  adata.obsm['X_umap']: UMAP (2D)")print(f"\nOutput files:")print(f"  Data: {OUTPUT_FILE}")print(f"  Figures: {FIGURES_DIR}/")print("\n" + "="*80)print("✓ Ready for Module 00.5: Regression")print("="*80)

## GATE**Why.** Module 02 requires BOTH `.X` log-normalized and `layers['counts']` raw. Checking here costs seconds; discovering it in module 02 costs a reload.

In [ ]:
# ⟨NEW⟩ exit gatewhy("01 GATE", "Is this object safe for module 02?")gate("layers['counts'] present", 'counts' in adata.layers)import numpy as np, scipy.sparse as sp_c = adata.layers['counts']_s = (_c.data[:1000] if sp.issparse(_c) else np.asarray(_c).flat[:1000])gate("counts are integers", bool(np.allclose(_s, np.round(_s))))gate("cell-type column present", cfg.cell_type_column in adata.obs,     cfg.cell_type_column)gate("no all-zero cells", int((np.asarray(_c.sum(axis=1)).ravel() == 0).sum()) == 0)print(f"\n  -> {adata.n_obs:,} cells x {adata.n_vars:,} genes ready for module 02")